# Pricing of options by Monte Carlo  

This notebook presents the basic functionalities of this modular Monte Carlo option pricing engine.

We start by generating geometric Brownian motion (GBM) paths under the risk-neutral measure. 
Next, we use these paths to estimate European call and put prices and compare them against the analytical Black–Scholes formula. 

In [ ]:
# for auto reloading  
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

# for importing from parent folder 
import sys
sys.path.append('..')

# import necessary packages 
import numpy as np
import matplotlib.pyplot as plt 
# import scipy.stats as st

import stoch_models
import numerical_schemes
import path_simulation
import option_contracts
# import estimators
import pricing_engine
import analytical_results


## Notation

| Notation | Explanation |
|----------|-------------|
| $S_0$ | Initial stock price |
| $K$ | Strike price |
| $T$ | Maturity |
| $r$ | Constant risk-free interest rate |
| $\sigma$ | Constant GBM volatility |
| $M$ | Number of time steps |
| $N$ | Number of paths |
| $S(t_i)$ | Exact stock price at time $t_i$ |
| $\widehat S_i^{(j)}$ | Numerical approximation for path $j$ at time $t_i$ |
| $W_t$ | Wiener process under the risk-neutral measure |
| $\Delta t$ | Time-step size $T/M$ |
| $\Delta W_i$ | Brownian increment $W(t_{i+1})-W(t_i)$ |
| $V(t_i,S)$ | Option value at time $t_i$ when the stock price is $S$ |
| $\Phi(T,S_T)$ | Payoff at maturity, with $\Phi(T,S_T)=V(T,S_T)$ |
## The Monte Carlo Pricing Algorithm

Let us recall the steps of Monte Carlo option pricing.

1. Partition $[0,T]$ using $t_i=iT/M$, with $i=0,\ldots,M$ and $\Delta t=T/M$.
2. Generate stock paths under the risk-neutral measure, with path index $j=1,\ldots,N$.
3. Compute the payoff $\Phi(T,S_T^{(j)})$ at maturity for every path.
4. Estimate the option value
   $$
   V(0,S_0)=e^{-rT}\mathbb E^{\mathbb Q}[\Phi(T,S_T)]
   $$
   using the discounted sample average
   $$
   \widehat V_N=e^{-rT}\frac1N\sum_{j=1}^N\Phi(T,S_T^{(j)}).
   $$
5. Estimate the standard error of the price estimator and construct an approximate 95% confidence interval.

<!-- For Euler and Milstein, the terminal values are numerical approximations, introducing discretization bias in addition to sampling error. For European options under GBM, we can also simulate terminal values directly without generating full paths. -->
## Generate GBM paths

Let us start by generating paths, which follow a geometric Brownian motion (GBM) dynamics 
$$
dS_t=rS_t\,dt+\sigma S_t\,dW_t.
$$
which has the exact solution
$$
S(t_{i+1})=S(t_i)\exp\left[
\left(r-\frac12\sigma^2\right)\Delta t+\sigma\Delta W_i
\right].
$$ 

<!-- 
Price processes can be approximated by different kind of schemes, which use the Ito-Taylor expansion. We refere to the stat_test_notebook.ipynb notebook for an introduction to the Euler and Milstein scheme. There we will use the exact solution of the GBM process to benchmark the approximative schemes. Here we will consider only the GBM exact solution paths.  -->

In [ ]:
# define model and simulation parameters
model = stoch_models.GBM(
    r=0.05,     # interest rate 
    sigma=0.2   # volatility
)

# set parameters
S_0 = 10    # inital price
N = 5000    # number of paths 
M = 2000    # number of time steps 
T = 1.0     # maturity

# define time grid for discretization
times = np.linspace(0, T, M + 1)

# random numbers
rng = np.random.default_rng(42)
# rng = np.random
Z = rng.standard_normal(size=(N, M))

# Exact GBM simulation
paths = path_simulation.simulate(
    model        = model,
    S_0          = S_0,
    times        = times,
    normal_draws = Z,
    scheme       = numerical_schemes.exact_step,
)


In [ ]:
S = paths.paths
times = paths.times

paths_to_show = 5

fig, (ax_paths, ax_hist) = plt.subplots(
    1, 2,
    figsize=(11, 4),
    constrained_layout=True,
)

# Show a few exact paths
ax_paths.plot(
    times,
    S[:paths_to_show].T,
    linewidth=1,
)

ax_paths.set(
    title="Exact GBM sample paths",
    xlabel="Time",
    ylabel="Stock price",
)

# Terminal distribution
ax_hist.hist(
    # S[:, -1],
    paths.terminal_prices,
    bins=40,
    density=True,
    color="tab:blue",
    alpha=0.7,
)

ax_hist.set(
    title="Terminal stock price distribution",
    xlabel=r"$S_T$",
    ylabel="Density",
)

plt.show()


## Pricing of European call and put options 

Here we show how to estimate the price of a call option and then compare it to the analytical Black-Scholes formula

In [ ]:
model = stoch_models.GBM(r=0.05, sigma=0.2)

option = option_contracts.EuropeanCall(
    strike=9,
    maturity=1.0,
)

S_0 = 10

# Analytical price
bs_price = analytical_results.black_scholes_price(
    model=model,
    product=option,
    S_0=S_0,
)

# Monte Carlo price
result = pricing_engine.price_option(
    model=model,
    product=option,
    scheme=numerical_schemes.exact_step,
    S_0=S_0,
    N=N,
    times=times,
    normal_draws=Z,
)

print(f"Black-Scholes price: {bs_price:.6f}")
print(f"Monte Carlo price:   {result.price:.6f}")
print(f"Pricing error:       {result.price - bs_price:.6f}")
print(f"Standard error:      {result.standard_error:.6f}")
print(f"95% CI lower:        {result.ci_lower:.6f}")
print(f"95% CI upper:        {result.ci_upper:.6f}")


In [ ]:

price = result.price
se = result.standard_error

fig, ax = plt.subplots(
    figsize=(8, 2.5),
    constrained_layout=True,
)

# Approximate 95% confidence interval
ax.errorbar(
    price, 0,
    # xerr=1.96 * se,
    xerr=[[price - result.ci_lower],
          [result.ci_upper - price]],
    fmt="none",
    ecolor="tab:blue",
    capsize=6,
    label="Approx. 95% CI",
)

# Monte Carlo price ± 1 standard error
ax.errorbar(
    price, 0,
    xerr=se,
    fmt="o",
    markersize=12,
    color="tab:blue",
    elinewidth=5,
    label="MC price ± 1 SE",
)

# Analytical benchmark
ax.axvline(
    bs_price,
    color="black",
    linestyle="--",
    label="Black–Scholes",
)

ax.set(
    title=f"{type(option).__name__} price",
    xlabel="Option price",
    ylim=(-1, 1),
)

ax.set_yticks([])
ax.legend(loc="upper center", ncol=3, frameon=False)

plt.show()
